# TOTO Zero-Shot Forecast — DataDog's Time Series Foundation Model

This notebook demonstrates using **TOTO 2.0** (DataDog's time series foundation model)
with the M5 retail dataset.  TOTO is a decoder-only transformer pre-trained on ~2
trillion time series data points.  It requires **no training** — just feed it the
last N days of each series as context and get a 28-day forecast.

---

**References:**
- [GitHub](https://github.com/datadog/toto)
- [Blog](https://www.datadoghq.com/blog/datadog-time-series-foundation-model/)
- [Paper (2.0)](https://arxiv.org/abs/2605.20119)

**Requirements:**
Run from the repo root with both the notebook and toto dep groups:
```bash
make notebook-toto
# or equivalently:
# uv run --group notebook --group toto jupyter lab
```

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from m5.config import SETTINGS
from m5.models.toto import toto_forecast, toto_cv, build_toto_model
from m5.evaluation import compute_components, wrmsse_for_models

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

## 1. Load processed data

In [ ]:
long_path = SETTINGS.processed_dir / "long.parquet"
print(f"Loading {long_path}")
df = pd.read_parquet(long_path)
df["ds"] = pd.to_datetime(df["ds"])
print(f"{len(df):,d} rows, {df['unique_id'].nunique():,d} unique series")
print(f"Date range: {df['ds'].min().date()} → {df['ds'].max().date()}")

In [ ]:
# Sample a few series for visualisation (pick the same seeds every run)
rng = np.random.default_rng(42)
all_ids = df["unique_id"].drop_duplicates().to_numpy()
hero_ids = list(rng.choice(all_ids, size=4, replace=False))
hero = df[df["unique_id"].isin(hero_ids)].copy()
print(f"Hero series: {hero_ids}")
print(f"Rows per series:
{hero.groupby('unique_id').size()}")


## 2. Build the TOTO model

We use the 22M-parameter variant which balances speed and accuracy.
Larger models (313M, 1B, 2.5B) are available via HuggingFace.

In [ ]:
device = "cuda"  # or "cpu"
model = build_toto_model(device=device)
print(f"Model loaded on {next(model.parameters()).device}")

## 3. Zero-shot forecast on hero series

TOTO forecasts directly from the last N days of context — no fitting step.

In [ ]:
from m5.models.toto import toto_forecast

# Forecast 28 days ahead using the last 512 days as context
fcast = toto_forecast(
    hero,
    horizon=28,
    context_length=512,
    batch_size=4,
)
print(fcast.head())
print(f"\nForecast shape: {fcast.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()

for ax, uid in zip(axes, hero_ids):
    hist = hero[hero["unique_id"] == uid].sort_values("ds")
    fut = fcast[fcast["unique_id"] == uid].sort_values("ds")

    if hist.empty:
        ax.set_title(f"{uid} (no data)")
        continue
    if fut.empty:
        ax.set_title(f"{uid} (no forecast)")
        continue

    cutoff = hist["ds"].max()
    ax.plot(hist["ds"], hist["y"], label="History", color="#1f77b4", alpha=0.7)
    ax.plot(fut["ds"], fut["TOTO"], label="TOTO forecast", color="#ff7f0e", linewidth=2)
    ax.axvline(x=cutoff, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(uid, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylabel("Sales")

plt.suptitle("TOTO Zero-Shot Forecast — 28-Day Horizon", fontsize=14)
plt.tight_layout()
plt.show()


## 4. Cross-validation

Run 3-fold rolling-origin CV to evaluate TOTO's zero-shot performance.
Each window uses data up to the cutoff as context and forecasts 28 days ahead.

In [ ]:
# Subsampled CV for speed (500 series, 1 window)
n_series = 500
uids = df["unique_id"].drop_duplicates().sample(n=n_series, random_state=42)
sub = df[df["unique_id"].isin(uids)].copy()
print(f"Subsampled: {len(sub):,d} rows, {sub['unique_id'].nunique():,d} series")

cv_df = toto_cv(
    sub,
    h=28,
    n_windows=1,
    context_length=512,
    batch_size=32,
)
print(f"CV result: {len(cv_df):,d} rows")
print(cv_df.head())

In [ ]:
# Score with WRMSSE
train_pre_cv = sub[sub["ds"] < cv_df["ds"].min()]
components = compute_components(train_pre_cv)
truth = cv_df[["unique_id", "ds", "y"]]
scores = wrmsse_for_models(truth, cv_df, components)
print("WRMSSE:")
print(scores.to_string())

## 5. Compare TOTO vs baseline models

Run stats (Theta/AutoETS/SeasonalNaive) CV on the same subsample for comparison.

In [ ]:
from m5.cv import stats_cv

stats_cv_df = stats_cv(sub, h=28, n_windows=1)

# Merge with TOTO results for scoring
merged = cv_df[["unique_id", "ds", "cutoff", "y", "TOTO"]].merge(
    stats_cv_df[["unique_id", "ds", "cutoff", "Theta", "AutoETS", "SeasonalNaive"]],
    on=["unique_id", "ds", "cutoff"],
)
components = compute_components(train_pre_cv)
comparison = wrmsse_for_models(
    merged[["unique_id", "ds", "y"]],
    merged,
    components,
)
print("\n=== WRMSSE Leaderboard ===")
print(comparison.to_string())

## 6. Full-data forecast (via CLI)

Run from the terminal (requires `--group toto` for the dependency):

```bash
# CV on the full dataset (this will take a while)
M5_N_SERIES=2000 M5_N_WINDOWS=1 uv run --group toto m5 cv toto --horizon 28

# Future forecast
uv run --group toto m5 forecast toto --horizon 28

# Score against baselines
uv run m5 score --model stats --model lgbm --model toto
```